In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "kano2014cross")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "2014_anibeh_first_look_edited.csv")
complete_path_2 = os.path.join(original_data_pathway, "2014_anibeh_total_look_edited.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)

experiment_import = [[df1, 'first_look'],
                    [df2, 'total_look']]

for x,y in experiment_import:
    x['experiment_name'] = y


In [3]:
data_frames=[df1, df2]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"subject": "ape",
        "sex":"sex_original"}, inplace=True)
    x['ape'] = x['ape'].str.rstrip()
    x['study_id']="kano2014cross"
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)


In [4]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')


In [5]:
fulldf.columns = fulldf.columns.str.replace(' ', '_')
fulldf.dropna(subset=['species'], inplace=True)
# fulldf.columns

fulldf.rename(columns={"ape": "participant",
                       "age":"age_original"}, inplace=True)
fulldf['experiment']=''

In [6]:
fulldf=fulldf[['study_id','experiment', 'experiment_name','participant', 'age_original', 'sex', 'species', 
       'own_species_model_target', 'own_species_model_distractor',
       'human_model_target', 'human_model_distractor',
       'other_species_model_target', 'other_species_model_distractor']]

exp1 = fulldf.drop(columns=['other_species_model_target', 'other_species_model_distractor'])
exp1 = exp1.assign(experiment='1')
exp3 = fulldf.drop(columns=['own_species_model_target', 'own_species_model_distractor',
                                'human_model_target', 'human_model_distractor'])
exp3 = exp3.assign(experiment='3')

experiments = [[exp1, 'kano2014cross_exp1'],
                [exp3, 'kano2014cross_exp3']]

for x,y in experiments:
    x = x.dropna(axis=1, how='all')## drop empty rows/columns
    comp_out_path_stand = os.path.join(out_pathway, y+'_standardized.csv')
    x.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)
    ##glossaries
    names = x.columns.tolist()
    df = pd.DataFrame(names)
    df = df.rename(columns={0: "column_name"})
    df["description"] = ""
    studyID_glossary=df[["column_name", "description"]]

    comp_out_path_glossary = os.path.join(out_pathway, y+'_glossary.csv')
    studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)


